In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import wandb
import os
from collections import OrderedDict

os.environ["WANDB_MODE"] = "offline"
wandb.init(project="vat sa-v-module", config={
    "model": "efficientnet_b0",
    "lr": 1e-3,
    "batch_size": 256,      # smaller batch often better for small images
    "epochs": 20,
    "pretrained": True
})

# === STRONG DATA AUGMENTATION (this makes a huge difference) ===
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),   # EfficientNet expects ~224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],       # ImageNet stats!
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True, num_workers=8, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=8, pin_memory=True)

# === VATSA VISUAL ENCODER (reusable!) ===
class VATSA_VisualEncoder(nn.Module):
    def __init__(self, embedding_dim=512, freeze_backbone=True):
        super().__init__()
        # Load PRETRAINED ImageNet weights — this is the game changer
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.backbone = efficientnet_b0(weights=weights)
        
        # Remove classifier, keep features
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        # Projection to shared VATSA latent space (512-dim as in your concept paper)
        self.projection = nn.Linear(num_features, embedding_dim)
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            # Only train the projection head first (fast + stable)
            for param in self.projection.parameters():
                param.requires_grad = True
    
    def forward(self, x):
        # x shape: (B, 3, 224, 224)
        features = self.backbone(x)                    # (B, 1280)
        embedding = self.projection(features)          # (B, 512) — shared latent
        return embedding

device = "cuda" if torch.cuda.is_available() else "cpu"
model = VATSA_VisualEncoder(embedding_dim=512, freeze_backbone=True).to(device)

# Optional: after a few epochs you can unfreeze last layers for fine-tuning
# for param in model.backbone.features[6:].parameters():  # last few stages
#     param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler()

# Simple LR scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# === TRAINING LOOP (cleaner) ===
for epoch in range(20):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            # For classification during training we can add a temp head or use projection + linear
            # For simplicity, we'll add a small classifier on top of embedding during training only
            emb = model(images)
            outputs = nn.Linear(512, 10).to(device)(emb)   # temp head
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    wandb.log({"epoch": epoch, "train_loss": avg_loss})
    print(f"Epoch {epoch+1}/20 - Loss: {avg_loss:.4f}")

# Save the encoder (backbone + projection)
torch.save({
    'model_state': model.state_dict(),
    'embedding_dim': 512
}, "vat sa_visual_encoder_cifar10.pth")

print("✅ VATSA Visual Encoder saved!")